# Tutorial 6: Difference between Causation and Association, and Model Diagnostics

#### Lecture and Tutorial Learning Goals:
After completing this week's lecture and tutorial work, you will be able to:

1. Give an example of a real problem that aims to test a causal relationship between variables.
2. Give an example of a real problem the model can only establish an association between the response and the input variables.
3. Discuss how the desired goal of generative modelling is usually to make causal claims however we cannot often/easily do so (e.g., in particular in the context of observational studies).
4. Discuss the role of confounders in causal inference.
5. Describe heteroscedasticity and the problem it presents to generative modeling.
6. Write a computer script to assess whether heteroscedasticity in a given data set, and if so, use practical solutions to manage it.
7. Describe colinearity and the problem it presents to generative modeling.
8. Write a computer script to assess whether collinearity exists between input variables in a given data set, and if so, use practical solutions to manage it.
9. Discuss what model diagnostics the data scientist can do by themselves and when the data scientist needs to consult a domain expert.

In [ ]:
# Run this cell before continuing.
library(tidyverse)
library(repr)
library(digest)
library(infer)
library(cowplot)
library(broom)
library(GGally)
library(car)
library(janitor)
library(scales)
source("tests_tutorial_06.R")

## 1. Multicollinearity in Practice

In this tutorial, we are going to re-examine the dataset `US_cancer_data`, containing cancer mortality rates and demographic and medical variables for American counties. Recall that the response $Y$, in this case, is `TARGET_deathRate`: a continuous variable that measures the cancer mortality per capita (for every 100,000 inhabitants), obtained from the years 2010-2016 as an average (further details available in [data.world](https://data.world/nrippner/ols-regression-challenge)).

In [ ]:
US_cancer_data <- read_csv("data/US_county_cancer_data.csv")

**Question 1.0**
<br>{points: 1}

For simplicity, we'll use the following subset of input variables from `US_cancer_data`:

- `incidenceRate`: average per capita (cases/100,000) cancer diagnoses in the county.
- `povertyPercent`: percent of the county's populace in poverty.
- `MedianAge`: median age of county's residents.
- `PctPrivateCoverage`: percent of county's residents with private health coverage.
- `medIncome`: median annual household income per county in American dollars.

Extract the 5 variables detailed above and the response variable, `TARGET_deathRate`, from `US_cancer_data`. Call the resulting dataset `US_county_sample`.

*Fill out those parts indicated with `...`, uncomment the corresponding code in the cell below, and run it.*

In [ ]:
# US_county_sample <- US_cancer_data %>%
#   ...(...)

# head(US_county_sample)

# your code here
fail() # No Answer - remove if you provide an answer

In [ ]:
test_1.0()

**Question 1.1**
<br>{points: 1}

Let's start by exploring (visually) the association between variables in the dataset.

Use the plotting function ggpairs(), from the library GGally, to generate a pair plot of ALL the variables found in `US_county_sample`. The ggplot() object's name will be `US_cancer_pair_plots`.

*Fill out those parts indicated with ..., uncomment the corresponding code in the cell below, and run it.*

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 12) # Adjust these numbers so the plot looks good in your desktop.

# US_cancer_pair_plots <- ... %>%
#   ...(progress = FALSE) +
#   theme(
#     text = element_text(size = 15),
#     plot.title = element_text(face = "bold"),
#     axis.title = element_text(face = "bold")
#   )
# US_cancer_pair_plots

# your code here
fail() # No Answer - remove if you provide an answer


In [ ]:
test_1.1()

**Question 1.2**
<br>{points: 1}

Based on the pairwise plots and correlation coefficients in `US_cancer_pair_plots`, do any input variables appear to be highly positive and/or negatively correlated? Which ones?

Recall that a correlation coefficient equal to $1$ means a perfectly positive linear relation and $-1$ a perfectly negative linear relation.

> *Your answer goes here.*

DOUBLE CLICK TO EDIT **THIS CELL** AND REPLACE THIS TEXT WITH YOUR ANSWER.

**Question 1.3**
<br>{points: 1}

Another way to visualize pairwise correlation coefficients between all the five input variables is to use a heatmap. 

Creating this visualization will require prior data wrangling on `US_county_sample`. Firstly, let us create a *melted* correlation matrix with the columns corresponding to the input variables from `US_county_sample` and name it `corr_matrix_US_county_sample`.

> **Hint:** You can use the function `cor()` for this purpose.

The code below will yield a data frame called `corr_matrix_US_county_sample` with three columns:

- `var1`: first input variable.
- `var2`: second input variable.
- `corr`: correlation coefficient between `var1` and `var2`.

*Fill out those parts indicated with `...`, uncomment the corresponding code in the cell below, and run it.*

In [ ]:
# corr_matrix_US_county_sample <- US_county_sample %>%
#   ungroup() %>%
#   select(
#     ...
#   ) %>%
#   ...() %>%
#   as.data.frame() %>%
#   rownames_to_column("var1") %>%
#   pivot_longer(-var1, names_to = "var2", values_to = "corr")
# corr_matrix_US_county_sample

# your code here
fail() # No Answer - remove if you provide an answer

In [ ]:
test_1.3()

**Question 1.4**
<br>{points: 1}

To create a proper $5 \times 5$ correlation matrix on top of a heatmap using `corr_matrix_US_county_sample`, the function `geom_tile()` from `ggplot2` is necessary. Create a plot called `plot_corr_matrix_US_county_sample` with the following characteristics:

- It would have the style of a heatmap where the correlation coefficient gives the colour scale. You have to use `scale_fill_distiller()` to select the colour palette (`YlOrRd`, preferably) and adjust the correlation limits on the scale (between `-1` and `1`).
- The scale legend's title has to be indicated as **Correlation Coefficient**.
- Include the correlations from `corr_matrix_US_county_sample` in the corresponding matrix cells **rounded to 2 decimal places**. Use `geom_text()`.

*Fill out those parts indicated with `...`, uncomment the corresponding code in the cell below, and run it.*

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 10) # Adjust these numbers so the plot looks good in your desktop.

# plot_corr_matrix_US_county_sample <- corr_matrix_US_county_sample %>%
#   ggplot(aes(..., ...)) +
#   ...(aes(fill = ...), color = "white") +
#   ...("Correlation Coefficient \n",
#     palette =  ...,
#     direction = 1, limits = ...
#   ) +
#   labs(x = "", y = "") +
#   theme_minimal() +
#   theme(
#     axis.text.x = element_text(
#       angle = 45, vjust = 1,
#       size = 18, hjust = 1
#     ),
#     axis.text.y = element_text(
#       vjust = 1,
#       size = 18, hjust = 1
#     ),
#     legend.title = element_text(size = 18, face = "bold"),
#     legend.text = element_text(size = 18),
#     legend.key.size = unit(2, "cm")
#   ) +
#   coord_fixed() +
#   ...(aes(..., ..., label = round(..., ...)), color = "black", size = 6)
# plot_corr_matrix_US_county_sample

# your code here
fail() # No Answer - remove if you provide an answer

In [ ]:
test_1.4()

**Question 1.5**
<br>{points: 1}
Let's examine how the multicollinearity problem affects the least squares estimators. Using `US_county_sample`, estimate an MLR called `MLR_US_county_sample` relating the response `TARGET_deathRate` to all the input variables selected.

*Tip*: in `lm`, you can use `.` at the RHS to avoid typing all input variables

> lm(response ~ ., data)

Obtain the estimated coefficients and standard errors using `tidy()`. Store the results in `MLR_US_county_sample_results`.


*Fill out those parts indicated with `...`, uncomment the corresponding code in the cell below, and run it.*

In [ ]:
# MLR_US_county_sample <- ...(..., ...)

# MLR_US_county_sample_results <- ...(...) %>% mutate_if(is.numeric, round, 2)
# MLR_US_county_sample_results

# your code here
fail() # No Answer - remove if you provide an answer

In [ ]:
test_1.5()

**Question 1.6**
<br>{points: 1}

Some of the input variables are highly correlated and will be a problem for the least square estimates of the regression coefficients and their standard errors. A problem known as *multicollinearity*.

We can use the Variance Inflation Factor (VIF) to help quantify multicollinearity in a dataset. VIF ranges from $1$ (no multicollinearity) to infinity. The smallest the VIF, the less multicollinearity. 

Roughly speaking, the VIF of a given input variable is the ratio of the *variance* of its estimated regression coefficient in the full MLR divided by the *variance* of its estimated regression coefficient coming from another simple linear regression model (involving the same input variable).

Calculate the VIF for your model `MLR_US_county_sample`. Assign the results to an object `VIF_MLR_US_county_sample`.

> **Hint:** A `vif()` function is available as part of the `car` package.

*Fill out those parts indicated with `...`, uncomment the corresponding code in the cell below, and run it.*

In [ ]:
# VIF_MLR_US_county_sample <- ...(...)
# round(VIF_MLR_US_county_sample, 3)

# your code here
fail() # No Answer - remove if you provide an answer

In [ ]:
test_1.6()

**Question 1.7**
<br>{points: 1}

Based on the results found in `plot_corr_matrix_US_county_sample` and `VIF_MLR_US_county_sample`, answer the following questions:

**1.7.0.** Which pair of variables have the highest VIFs?

**A.** `incidenceRate`.

**B.** `avgAnnCount`.

**C.** `avgDeathsPerYear`.

**D.** `povertyPercent`.

**E.** `MedianAge`.

**F.** `PctPrivateCoverage`.

**G.** `medIncome`.

*Assign your answer to the object `answer1.7.0`. Your answer must be a single string indicating the correct options **in alphabetical order** and surrounded by quotes (e.g., `"FG"` indicates you are selecting these two options).*

**1.7.1.** Do this pair corresponds to the highest absolute value from all the pairwise correlations shown in the heatmap?

**A.** Yes.

**B.** No.

*Assign your answer to an object called `answer1.7.1`. Your answer should be a single character surrounded by quotes.*


In [ ]:
# answer1.7.0 <- ...
# answer1.7.1 <- ...

# your code here
fail() # No Answer - remove if you provide an answer


In [ ]:
test_1.7.0()
test_1.7.1()

**Question 1.8**
<br>{points: 1}

One way to deal with multicollinearity is dropping those input variables associated with the largest VIFs and pairwise correlation coefficients. We now estimate a reduced MLR using only a subset of the input variables in `US_county_sample`, call it `red_MLR_US_county_sample`, as follows:

- From the pair of input variables with the highest absolute correlation (positive or negative) in `plot_corr_matrix_US_county_sample`, only use the one with the smallest VIF found in `VIF_MLR_US_county_sample`.
- Add **the rest** of input variables found in `US_county_sample`.

Hence, `red_MLR_US_county_sample` will have four input variables. Use `tidy()` to summarize results in `red_MLR_US_county_sample_results`

*Fill out those parts indicated with `...`, uncomment the corresponding code in the cell below, and run it.*

In [ ]:
# red_MLR_US_county_sample <- ...(..., ...)

# red_MLR_US_county_sample_results <- ...(...) %>% mutate_if(is.numeric, round, 2)
# red_MLR_US_county_sample_results

# your code here
fail() # No Answer - remove if you provide an answer

In [ ]:
test_1.8()

**Question 1.9**
<br>{points: 1}

Now, obtain the VIFs with `red_MLR_US_county_sample`. Assign the results to an object `VIF_red_MLR_US_county_sample`.

*Fill out those parts indicated with `...`, uncomment the corresponding code in the cell below, and run it.*

In [ ]:
# VIF_red_MLR_US_county_sample <- ...(...)
# round(VIF_red_MLR_US_county_sample, 3)

# your code here
fail() # No Answer - remove if you provide an answer

In [ ]:
test_1.9()

**Question 1.10**
<br>{points: 1}

James et al. (2013) in [*An Introduction to Statistical Learning*](https://www.statlearning.com/) (see Section 3.3.3 in Subsection Collinearity) indicate that, as a rule of thumb, a VIF value that exceeds 5 or 10 poses problems in terms of multicollinearity. 

Based on the results found in `VIF_red_MLR_US_county_sample`, `MLR_US_county_sample_results`, and `red_MLR_US_county_sample_results`, are the VIFs in `VIF_red_MLR_US_county_sample` below that threshold? 

**A.** Yes.

**B.** No.

*Assign your answer to an object called `answer1.10`. Your answer should be a single character surrounded by quotes.*

In [ ]:
# answer1.10 <- ...

# your code here
fail() # No Answer - remove if you provide an answer

In [ ]:
test_1.10()

## 2. Confounding and Simpson's Paradox

To wrap up the topic of confounding variables, we will relate it to the concept of Simpson's paradox. The Simpson's paradox appears when we detect a given trend between the response and input of interest once we consider a second input variable (which disaggregates our data). Nonetheless, this trend changes if we do not use this second input in our analysis, leading to aggregating our data to just the response and input of interest. Note that this second input is considered a confounder.

We will exemplify the Simpson's paradox via a common dataset found in the literature. This dataset is called `UCBAdmissions`. It contains aggregate data from 1973, in the form of six contingency tables, on applicants to graduate school at the University of California, Berkeley (UCB) for the six largest departments (labelled as `A`, `B`, `C`, `D`, `E`, and `F` in the column `Dept`). Each table shows applicants' absolute frequencies by admission status (`Admitted` or `Rejected` in the column `Admit`) and a `Gender` column (`Male` or `Female`). It has 4526 applicants in total.

> **Heads-up:** A contingency table is merely a cross-tabulation of two categorical variables. We place the levels of one variable in the rows and the levels of the second variable in the columns. The cells in this table indicate the counts corresponding to each level combination.

The raw dataset `UCBAdmissions`, as shown below, disaggregates the 4526 applicants in the form of contigency tables by department. 

In [ ]:
UCBAdmissions

We need to do some data wrangling on `UCBAdmissions` to change it to a tibble where each row is one of the 4526 applicants. Thus, we need `UCBAdmissions` to have three columns: `Admit`, `Gender`, and `Dept`. Run the cell below before continuing.

In [ ]:
 UCBAdmissions <- UCBAdmissions %>%
   as.data.frame() %>%
   as_tibble() %>%
   mutate(obs = map(Freq, ~ rep_len(1, .x))) %>%
   unnest(cols = c(obs)) %>%
   select(-Freq, -obs)

 head(UCBAdmissions)
 tail(UCBAdmissions)
 nrow(UCBAdmissions)


**Question 2.0**
<br>{points: 1}

[Bickel et al. (1975)](https://science.sciencemag.org/content/187/4175/398) used the dataset `UCBAdmissions` to illustrate Simpson's paradox, as we will see further. The authors claimed that just using the aggregated data of admission status and gender, **without considering the confounding variable `Dept`**, might result in a misleading conclusion of gender bias in admissions. Nonetheless, they further show that this overall conclusion is not statistically significant once the data is disaggregated by department unit. Let us check this fact.

Firstly, we will perform a graphical analysis of the overall data via stacked bar charts. Before doing this, compute the **overall counts and proportions on a `Gender` basis** by each level combination of `Gender` and `Admit` in a data frame called  `overall_UCBAdmissions.prop.summary`. This data frame will have four columns:

- `Gender`: a column where each of the four rows will be either `Male` or `Female`.
- `Admit`: a column where each of the four rows will be either `Admitted` or `Rejected`.
- `n`: a column where each of the four rows is the number of applicants corresponding to each level combination of `Gender` and `Admit`.
- `proportion`: a column where each of the four rows is the proportion of `Admitted` and `Rejected` applicants **within each category of `Gender`**.

*Fill out those parts indicated with `...`, uncomment the corresponding code in the cell below, and run it.*

In [ ]:
# overall_UCBAdmissions.prop.summary <- UCBAdmissions %>%
#   ...(..., ...) %>%
#   summarize(n = length(Gender)) %>%
#   ...(proportion = round(... / sum(...), 2))
# overall_UCBAdmissions.prop.summary

# your code here
fail() # No Answer - remove if you provide an answer

In [ ]:
test_2.0()

**Question 2.1**
<br>{points: 1}

We already pointed out that we will use stacked bar charts in our graphical analysis. This class of plots is ideal when we want to depict the relationship of two categorical variables, such as `Gender` and `Admit`. Create this `ggplot()` object whose name will be `overall_UCBAdmissions.stacked.bars`. Each bar on the $y$-axis will show the percentages (i.e., column `proportion` in `overall_UCBAdmissions.prop.summary`) of applicants in each category of `Admit` with the categories found in `Gender` on the $x$-axis. This class of bars will allow us to compare both levels of `Gender` in terms of the categories of `Admit`. Note that the function `percent_format()`, from library `scales`, changes the format of the $y$-axis text to percentages.

*Fill out those parts indicated with `...`, uncomment the corresponding code in the cell below, and run it.*

In [ ]:
# overall_UCBAdmissions.stacked.bars <- ggplot(overall_UCBAdmissions.prop.summary, aes(
#   x = ...,
#   y = ..., fill = ...
# )) +
#   ...(stat = "identity", width = 0.7, colour = "black", lwd = 0.1) +
#   geom_text(aes(label = ifelse(proportion >= 0.01, paste0(sprintf("%.0f", proportion * 100), "%"), "")),
#     position = position_stack(vjust = 0.5), colour = "black", fontface = "bold", size = 6
#   ) +
#   scale_y_continuous(labels = percent_format()) +
#   labs(y = "Percent", x = "Gender", fill = "") +
#   ggtitle(...) +
#   theme(
#     plot.title = element_text(face = "bold"),
#     text = element_text(size = 19),
#     axis.title = element_text(face = "bold"),
#     legend.text = element_text(size = 15, margin = margin(r = 0.5, unit = "cm")),
#     legend.title = element_text(size = 15, face = "bold"),
#     legend.position = "bottom"
#   ) +
#   guides(fill = guide_legend(title = "Admit Status"))
# overall_UCBAdmissions.stacked.bars

# your code here
fail() # No Answer - remove if you provide an answer

In [ ]:
test_2.1()

**Question 2.2**
<br>{points: 1}

Graphically, what can we say about the relationship between `Admit` and `Gender`? Is there any evidence of gender bias in admission practices during 1973 at UCB?

> *Your answer goes here.*

DOUBLE CLICK TO EDIT **THIS CELL** AND REPLACE THIS TEXT WITH YOUR ANSWER.

**Question 2.3**
<br>{points: 1}

The raw dataset `UCBAdmissions` showed the contingency tables by department unit. However, the overall table without considering any department is not provided. `R` offers different functions to build contingency tables with our wrangled data frame `UCBAdmissions`. For example, the function `tabyl()` from library `janitor` automatically constructs a contingency table (or set) when providing two or more categorical columns.

*Fill out those parts indicated with `...`, uncomment the corresponding code in the cell below, and run it.*

In [ ]:
# overall_cont_table_Admit_vs_Gender <- UCBAdmissions %>%
#  ...(..., ...)
# overall_cont_table_Admit_vs_Gender

# your code here
fail() # No Answer - remove if you provide an answer

In [ ]:
test_2.3()

**Question 2.4**
<br>{points: 1}

It is possible to assess whether there is a statistically significant dependency between the admission status and gender. Since both variabels are discrete, an appropriate statistical test is the Pearson's Chi-Squared test. We use this test to statistically evaluate the independence between the rows and columns of a contingency table (e.g., `overall_cont_table_Admit_vs_Gender`). 

What do we mean by "independence" for this case? It means that the value of the row variable does not have any relation with the values of the column variable (or the other way around). Therefore, the fact of getting admitted is not related to any specific gender. Like any other statistical test, we need to set up of pair of hypotheses. The null hypothesis ($H_0$) for Pearson's Chi-Squared test states that the `Admit` status is independent of `Gender`. The alternative hypothesis ($H_1$) is that the `Admit` status is associated with `Gender`. Both hypotheses can be put in plain words as follows:

$$
H_0: \text{there is no gender bias in admissions,} \\
H_a: \text{there is gender bias in admissions.}
$$

Use the function `chisq.test()` to perform a Pearson's Chi-Squared test using `overall_cont_table_Admit_vs_Gender`. Store your results in `overall_chisq_test_Admit_vs_Gender` which have to be reported in a tidy format using `glance()` from library `broom`.

*Fill out those parts indicated with `...`, uncomment the corresponding code in the cell below, and run it.*

In [ ]:
# overall_chisq_test_Admit_vs_Gender <- overall_cont_table_Admit_vs_Gender %>%
#   ...() %>%
#   ...()
# overall_chisq_test_Admit_vs_Gender

# your code here
fail() # No Answer - remove if you provide an answer

In [ ]:
test_2.4()

**Question 2.5**
<br>{points: 1}

Using a significance level $\alpha = 0.05$, state the statistical conclusion of this test in this specific context.

> *Your answer goes here.*

DOUBLE CLICK TO EDIT **THIS CELL** AND REPLACE THIS TEXT WITH YOUR ANSWER.

**Question 2.6**
<br>{points: 1}

What if we disaggregate the analysis on `UCBAdmissions` by department? Will this reverse the statistical conclusion of the overall analysis? If this happens, then we will have a case where Simpson's paradox comes into the scene with a confounding input variable: `Dept`.

Let us perform a graphical analysis of the disaggregated data via stacked bar charts *facetted by the department*. Before doing this, **for each level of `Dept`**, compute the **counts and proportions on a `Gender` basis** by each level combination of `Gender` and `Admit` in a data frame called  `dept_UCBAdmissions.prop.summary.Dept`. This data frame will have four columns:

- `Dept`: a column denoting each department. You will have four rows for each label: `A`, `B`, `C`, `D`, `E`, and `F`.
- `Gender`: a column where each department's four rows will be either `Male` or `Female`.
- `Admit`: a column where each department's four rows will be either `Admitted` or `Rejected`.
- `n`: a column where each department's four rows is the number of applicants corresponding to each level combination of `Gender` and `Admit`.
- `proportion`: a column where each department's four rows is the proportion of `Admitted` and `Rejected` applicants **within each category of `Gender`**.

*Fill out those parts indicated with `...`, uncomment the corresponding code in the cell below, and run it.*

In [ ]:
# dept_UCBAdmissions.prop.summary.Dept <- UCBAdmissions %>%
#   ...(..., ..., ...) %>%
#   summarize(n = length(Dept)) %>%
#   ...(proportion = round(... / sum(...), 2))
# dept_UCBAdmissions.prop.summary.Dept

# your code here
fail() # No Answer - remove if you provide an answer

In [ ]:
test_2.6()

**Question 2.7**
<br>{points: 1}

Create a stacked bar char whose name will be `dept_UCBAdmissions.stacked.bars`. Each bar on the $y$-axis will show the percentages (i.e., column `proportion` in `overall_UCBAdmissions.prop.summary`) of applicants in each category of `Admit` with the categories found in `Gender` on the $x$-axis. This class of bars will allow us to compare both levels of `Gender` in terms of the categories of `Admit`. Note that the function `percent_format()`, from library `scales`, changes the format of the $y$-axis text to percentages.

> **Heads-up:** The plot `dept_UCBAdmissions.stacked.bars` has to be facetted by department, check `facet_wrap()`.

*Fill out those parts indicated with `...`, uncomment the corresponding code in the cell below, and run it.*

In [ ]:
# dept_UCBAdmissions.stacked.bars <- ggplot(dept_UCBAdmissions.prop.summary.Dept, aes(
#   x = ...,
#   y = ..., fill = ...
# )) +
#   ...(stat = "identity", width = 0.7, colour = "black", lwd = 0.1) +
#   geom_text(aes(label = ifelse(proportion >= 0.03, paste0(sprintf("%.0f", proportion * 100), "%"), "")),
#     position = position_stack(vjust = 0.5), colour = "black", fontface = "bold", size = 6
#   ) +
#   scale_y_continuous(labels = percent_format()) +
#   labs(y = "Percent", x = "Gender", fill = "") +
#   ggtitle(...) +
#   theme(
#     plot.title = element_text(face = "bold"),
#     text = element_text(size = 19),
#     axis.title = element_text(face = "bold"),
#     legend.text = element_text(size = 15, margin = margin(r = 0.5, unit = "cm")),
#     legend.title = element_text(size = 15, face = "bold"),
#     legend.position = "bottom"
#   ) +
#   guides(fill = guide_legend(title = "Admit Status")) +
#   facet_wrap(. ~ ...)
# dept_UCBAdmissions.stacked.bars

# your code here
fail() # No Answer - remove if you provide an answer

In [ ]:
test_2.7()

**Question 2.8**
<br>{points: 1}

Graphically, what can we say about the relationship between `Admit` and `Gender` while controlling by `Dept`? Is there graphical evidence of gender bias in admission practices during 1973?

> *Your answer goes here.*

DOUBLE CLICK TO EDIT **THIS CELL** AND REPLACE THIS TEXT WITH YOUR ANSWER.

**Question 2.9**
<br>{points: 1}

Let us perform the Pearson's Chi-Squared test but with the data disaggregated **by department**. The null hypothesis ($H_0$) for each Pearson's Chi-Squared test states that the `Admit` status is independent of `Gender` in a given department. The alternative hypothesis ($H_1$) is that the `Admit` status is associated with `Gender` in a given department. Both hypotheses can be put in plain words as follows:

$$
H_0: \text{there is no gender bias in admissions in a given department,} \\
H_a: \text{there is gender bias in admissions in a given department.}
$$

Store your results in `dept_chisq_test_Admit_vs_Gender` with a row by department. We do not need to generate departmental contingency tables to use the `chisq.test()` function. We can directly use `UCBAdmissions` if we first group the data by department. Then we summarize the `chisq.test()` results in two columns:

- `test_statistic`: the statistic from each Pearson's Chi-Squared test.
- `p.value`: the $p$-value from each Pearson's Chi-Squared test.

> **Heads-up:** We will perform multiple testing in this case; thus, we need to adjust the $p$-values in the column `adj_p.value`. Use the Bonferroni correction.

*Fill out those parts indicated with `...`, uncomment the corresponding code in the cell below, and run it.*

In [ ]:
# dept_chisq_test_Admit_vs_Gender <- UCBAdmissions %>%
#   ...(...) %>%
#   ...(
#     test_statistic = round(chisq.test(..., ...)$..., 2),
#     p.value = round(chisq.test(..., ...)$..., 5)
#   ) %>%
#   mutate(adj_p.value = ...(p.value, method = ...))
# dept_chisq_test_Admit_vs_Gender

# your code here
fail() # No Answer - remove if you provide an answer

In [ ]:
test_2.9()

**Question 2.10**
<br>{points: 1}

Using a significance level $\alpha = 0.05$, state the statistical conclusions of these tests in this specific context.

> *Your answer goes here.*

DOUBLE CLICK TO EDIT **THIS CELL** AND REPLACE THIS TEXT WITH YOUR ANSWER.